In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import altair as alt
import seaborn as sns
from IPython.display import display
import numpy as np

In [ ]:
os.getcwd()

In [ ]:
NodeType = {
  "INVALID_NODE": 0,
  "METADATA_NODE": 1,
  "MEM_LOAD_NODE": 2,
  "MEM_STORE_NODE": 3,
  "COMP_NODE": 4,
  "COMM_SEND_NODE": 5,
  "COMM_RECV_NODE": 6,
  "COMM_COLL_NODE": 7,
}
node_types = list(NodeType.keys())

In [ ]:
def get_timings_df(df: pd.DataFrame) -> pd.DataFrame:
    df_issues = df.query("action == 'issue'").drop(columns="action")
    df_callbacks = df.query("action == 'callback'").drop(columns="action")
    return df_issues.merge(
        df_callbacks,
        on=["sys_id", "node_id", "node_name", "node_type"],
        suffixes=("_issue", "_callback"),
    ).assign(elapsed_time=lambda d: d["tick_callback"] - d["tick_issue"])

def plot_elapsed_times(df: pd.DataFrame, sys_id: int = 0, max_height: int =600) -> alt.Chart:
    df = df.query(f"sys_id == {sys_id}")
    unique_nodes = df["node_name"].nunique()
    chart_height = unique_nodes * 20
    return alt.Chart(df).mark_bar().encode(
        x=alt.X('elapsed_time:Q', title='Elapsed Time'),
        y=alt.Y('node_name:N', sort=alt.SortField(field='tick_issue', order='ascending'), title='Node Name'),
        tooltip=['node_name', 'elapsed_time', 'tick_issue']
    ).properties(
        width=600,
        height=min(chart_height, max_height),
        title='Elapsed Time by Node Name'
    ).configure_axis(
        labelFontSize=10
    ).interactive()

def get_overlapped_blocks(df: pd.DataFrame) -> dict[str, list[int]]:
    """df should be filtered by sys_id and node_type."""
    df_sorted = df.sort_values("tick_issue")
    blocks = {
        "start": [],
        "end": [],
    }
    prev_start, prev_end = 0, 0
    for _, row in df_sorted.iterrows():
        if row["tick_issue"] <= prev_end:
            # overlap
            # merge the two blocks with proper start and end times
            # update the prev variables
            # do not append the block until we know it is not overlapping with any other subsequent block
            prev_end = max(prev_end, row["tick_callback"])
        else:
            # there is no overlap, append the block and update the prev variables
            blocks["start"].append(prev_start)
            blocks["end"].append(prev_end)
            prev_start = row["tick_issue"]
            prev_end = row["tick_callback"]
            
    blocks["start"].append(prev_start)
    blocks["end"].append(prev_end)
    return blocks

def plot_overlapped_blocks(df: pd.DataFrame) -> alt.Chart:
    chart = alt.Chart(df).mark_bar().encode(
        x=alt.X('start:Q', title='Timestamp'),
        x2='end:Q',
        y=alt.Y('node_type:N', title='Node Type'),
        color='node_type:N',  # Different color for each node_type
    ).configure_axis(
        grid=False,  # Remove the grid lines
        ticks=False
    ).properties(
        height=200,
        width=800,
        title='Duration of Blocks by Node Type'
    )
    
    return chart.interactive()

def plot_one_npu(df, npu, plot_blocks=True, plot_times=True):
    print("npu: ", npu)
    df_0 = df.query(f"sys_id == {npu}")
    df_0_comp = pd.DataFrame.from_dict(get_overlapped_blocks(df_0.query("node_type == 4"))).assign(
        node_type="COMPUTATION"
    )
    df_0_comm = pd.DataFrame.from_dict(get_overlapped_blocks(df_0.query("node_type in (5, 6, 7)"))).assign(
        node_type="COMMUNICATION"
    )
    df_0_blocks = pd.concat([df_0_comp, df_0_comm])
    if plot_blocks:
        display(plot_overlapped_blocks(df_0_blocks))
    if plot_times:
        display(plot_elapsed_times(df, max_height=600))

def plot_all_npus(df):
    for npu in range(64):
        df_0 = df.query(f"sys_id == {npu}")
        plot_one_npu(df, npu, plot_blocks=True, plot_times=False)

def plot_roofline(df, beta=5e10, pi=1e15):
    chart = alt.Chart(df).mark_circle().encode(
        x=alt.X('operational_intensity:Q'),
        y=alt.Y('perf:Q'),
        size=alt.value(100),
        #tooltip=['node_id', 'num_ops', 'tensor_size', 'perf', 'operational_intensity', 'elapsed_time']  # Show all columns
    ).properties(
        title='Performance vs Operational Intensity',
        width=600,
        height=400
    ).interactive()
    display(chart)

def plot_roofline(df, beta=2000, pi=300):
    # Compute intersection point
    beta = beta * (10 ** 9)
    pi = pi 
    I_c = pi / beta
    P_c = pi

    x_min = max(0, df['operational_intensity'].min())
    x_max = df['operational_intensity'].max() * 1.1  # Add 10% headroom
    x_vals = np.linspace(x_min, x_max, 200)


    # Create dataframes for the roofline model lines
    roofline_data = pd.DataFrame({
        'operational_intensity': x_vals,
        'beta_line': (beta * x_vals)/1e12,
        'pi_line': [pi] * len(x_vals)
    })

    # Base scatter plot
    chart = alt.Chart(df).mark_circle().encode(
        x=alt.X('operational_intensity:Q', title='Operational Intensity (FLOPs/byte)'),
        y=alt.Y('perf:Q', title='Performance (TFLOPs/sec)'),
        size=alt.value(100),
        tooltip=list(df.columns)
    ).properties(
        title='Roofline Model: Performance vs Operational Intensity',
        width=600,
        height=400
    )

    # Bandwidth line (sloped)
    beta_line = alt.Chart(roofline_data).mark_line(color='red').encode(
        x='operational_intensity:Q',
        y='beta_line:Q'
    )

    # Peak performance line (horizontal)
    pi_line = alt.Chart(roofline_data).mark_line(color='green').encode(
        x='operational_intensity:Q',
        y='pi_line:Q'
    )

    # Intersection point marker
    intersect_point = pd.DataFrame({
        'operational_intensity': [I_c],
        'perf': [P_c]
    })
    intersection = alt.Chart(intersect_point).mark_point(color='black', shape='cross', size=200).encode(
        x='operational_intensity:Q',
        y='perf:Q'
    )

    final_chart = (chart + beta_line + pi_line).interactive()
    display(final_chart)


# GPT 3 1300M 2D Torus

In [ ]:
df = pd.read_csv("output/GPT_3_1300M/2D_Torus/1_8_2_4_0_trace.csv")
df = get_timings_df(df)
plot_one_npu(df, 0)

In [ ]:
df = pd.read_csv("output/GPT_3_1300M/2D_Torus/4_2_2_4_0_trace.csv")
df = get_timings_df(df)
plot_one_npu(df, 0)
#plot_all_npus(df)

In [ ]:
df = pd.read_csv("output/GPT_3_1300M/2D_Torus/2_1_32_1_0_trace.csv")
df = get_timings_df(df)
#plot_one_npu(df, 0)
plot_all_npus(df)

In [ ]:
df = pd.read_csv("output/GPT_3_1300M/2D_Torus/4_1_16_1_0_trace.csv")
df = get_timings_df(df)
#plot_one_npu(df, 0)
plot_all_npus(df)

In [ ]:
df = pd.read_csv("output/GPT_3_1300M/2D_Torus/4_8_2_1_0_trace.csv")
df = get_timings_df(df)
plot_one_npu(df, 0)
#plot_all_npus(df)

In [ ]:
import plotly.graph_objects as go
import numpy as np

def plot_3d_roofline(df, beta=2000, pi=300):
    # Convert to SI units
    beta = beta * 1e9  # GB/s to B/s
    pi = pi * 1e12           # TFLOP/s

    # Sort by issue_tick for orderly processing
    df_sorted = df.sort_values(by='issue_tick')
    df_sorted['perf'] = df_sorted['perf']* 1e12
    # Set grid ranges based on data
    op_intensity_min = df_sorted['operational_intensity'].min()
    op_intensity_max = df_sorted['operational_intensity'].max()
    issue_tick_min = df_sorted['issue_tick'].min()
    issue_tick_max = df_sorted['issue_tick'].max()

    # Create grid for operational intensity (OI) and issue_tick (time)
    op_intensity = np.linspace(op_intensity_min, op_intensity_max, 50)
    issue_tick = np.linspace(issue_tick_min, issue_tick_max, 50)
    OI_grid, T_grid = np.meshgrid(op_intensity, issue_tick)

    # Compute Bandwidth bound surface (perf = beta * I)
    perf_bandwidth = beta * OI_grid

    # Compute Compute limit surface (perf = pi)
    perf_compute = np.full_like(OI_grid, pi)

    fig = go.Figure()

    # Scatter original data points
    fig.add_trace(go.Scatter3d(
        x=df_sorted['operational_intensity'],
        y=df_sorted['issue_tick'],
        z=df_sorted['perf'],
        mode='lines+markers',
        marker=dict(
            size=3,
            color=df_sorted['issue_tick'],
            colorscale='Viridis',
            opacity=0.4,
            showscale=False
        ),
        customdata=np.stack((df_sorted['node_id'],), axis=-1),
        hovertemplate=
            "Node ID: %{customdata[0]}<br>" +
            "Op Intensity: %{x:.2f}<br>" +
            "Perf: %{z:.2e}<br>" +
            "Issue Tick: %{y}<extra></extra>",
        name='Original Points'
    ))

    # Add Bandwidth bound surface
    fig.add_trace(go.Surface(
        x=OI_grid,
        y=T_grid,
        z=perf_bandwidth,
        colorscale=[[0, 'red'], [1, 'red']],
        opacity=0.3,
        showscale=False,
        name='Bandwidth Bound'
    ))

    # Add Compute limit surface
    fig.add_trace(go.Surface(
        x=OI_grid,
        y=T_grid,
        z=perf_compute,
        colorscale=[[0, 'green'], [1, 'green']],
        opacity=0.3,
        showscale=False,
        name='Compute Limit'
    ))

    # Configure 3D axes and optionally set axis ranges to match your data
    fig.update_layout(
        scene=dict(
            xaxis=dict(title='Operational Intensity (FLOPs/byte)', range=[op_intensity_min, op_intensity_max]),
            yaxis=dict(title='Issue Tick', range=[issue_tick_min, issue_tick_max]),
            zaxis=dict(title='Performance (FLOPs/sec)',  range=[0, pi])
        ),
        title='3D Roofline Model',
        width=900,
        height=800
    )

    fig.show()


In [ ]:
def add_comm_points(df, npu = 0):
    df_single_npu = df.query(f"sys_id == {npu}")
    df_single_npu = df_single_npu.drop(df.columns[0], axis=1)
    df_single_npu['perf'] = df_single_npu['perf'] / 1e+12
    df_single_npu['operational_intensity'] = df_single_npu['operational_intensity']
    new_rows = []

    df_sorted = df_single_npu.sort_values('issue_tick').reset_index(drop=True)
    # Iterate over the group, skipping the first and last row
    for i in range(1, len(df_sorted)):
        prev_row = df_sorted.iloc[i - 1]
        current_row = df_sorted.iloc[i]

        # Prepare the new row as specified
        new_row = {
            'sys_id': current_row['sys_id'],
            'node_id': 0,
            'node_name': prev_row['node_name'] + '_comm_' + current_row['node_name'],
            'num_ops': 0,
            'tensor_size': 0,
            'perf': 0,
            'operational_intensity': 0,
            'elapsed_time': current_row['issue_tick'] - prev_row['callback_tick'],
            'issue_tick': prev_row['callback_tick'],
            'callback_tick': current_row['issue_tick']
        }
        new_rows.append(new_row)

    # Append the new rows to the original dataframe
    df_new = pd.DataFrame(new_rows)
    df_combined = pd.concat([df_single_npu, df_new], ignore_index=True)

    # Sort the final dataframe by sys_id and issue_tick
    df_combined_sorted = df_combined.sort_values(['sys_id', 'issue_tick']).reset_index(drop=True)

    df_combined_sorted = df_combined_sorted[df_combined_sorted['elapsed_time'] != 0]

    # Save to a new CSV, or overwrite as needed
    #df_combined_sorted.to_csv("1_1_16_4_0_roofline_with_comm.csv", index=False)

    return df_combined_sorted

In [ ]:
def expand_df_and_average(df, time_window=50000): 
    new_rows = []
    buffer_row = None
    remaining_time = 0
    
    for i in range(len(df)):
        row = df.iloc[i].copy()
        
        # If we have a buffer row from previous iteration
        if buffer_row is not None:
            # Calculate how much time we need to complete the window
            time_needed = time_window - remaining_time
            
            if row['elapsed_time'] >= time_needed:
                # We can complete the window
                # Create a new row that completes the window
                new_row = buffer_row.copy()
                new_row['elapsed_time'] = time_window
                new_row['callback_tick'] = new_row['issue_tick'] + time_window
                
                # Calculate weighted averages for performance metrics
                weight_prev = remaining_time / time_window
                weight_curr = time_needed / time_window
                new_row['perf'] = (buffer_row['perf'] * weight_prev) + (row['perf'] * weight_curr)
                new_row['operational_intensity'] = (buffer_row['operational_intensity'] * weight_prev) + (row['operational_intensity'] * weight_curr)
                
                new_rows.append(new_row)
                
                # Update the current row
                row['elapsed_time'] -= time_needed
                row['issue_tick'] += time_needed
                
                # Process the remaining time in the current row
                remaining_full_windows = int(row['elapsed_time'] // time_window)
                
                # Add full windows
                for j in range(remaining_full_windows):
                    window_row = row.copy()
                    window_row['elapsed_time'] = time_window
                    window_row['issue_tick'] = row['issue_tick'] + (j * time_window)
                    window_row['callback_tick'] = window_row['issue_tick'] + time_window
                    new_rows.append(window_row)
                
                # Calculate the remaining time after full windows
                remaining_time = row['elapsed_time'] % time_window
                if remaining_time > 0:
                    # Store the remaining part for the next iteration
                    buffer_row = row.copy()
                    buffer_row['elapsed_time'] = remaining_time
                    buffer_row['issue_tick'] = row['issue_tick'] + (remaining_full_windows * time_window)
                    buffer_row['callback_tick'] = buffer_row['issue_tick'] + remaining_time
                else:
                    buffer_row = None
                    remaining_time = 0
            else:
                # We can't complete the window yet
                remaining_time += row['elapsed_time']
                
                # Update buffer row with weighted averages
                total_time = buffer_row['elapsed_time'] + row['elapsed_time']
                weight_buffer = buffer_row['elapsed_time'] / total_time
                weight_row = row['elapsed_time'] / total_time
                
                buffer_row['perf'] = (buffer_row['perf'] * weight_buffer) + (row['perf'] * weight_row)
                buffer_row['operational_intensity'] = (buffer_row['operational_intensity'] * weight_buffer) + (row['operational_intensity'] * weight_row)
                buffer_row['elapsed_time'] = total_time
                buffer_row['callback_tick'] = buffer_row['issue_tick'] + total_time
        else:
            # No buffer row, process the current row directly
            full_windows = int(row['elapsed_time'] // time_window)
            
            # Add full windows
            for j in range(full_windows):
                window_row = row.copy()
                window_row['elapsed_time'] = time_window
                window_row['issue_tick'] = row['issue_tick'] + (j * time_window)
                window_row['callback_tick'] = window_row['issue_tick'] + time_window
                new_rows.append(window_row)
            
            # Calculate the remaining time
            remaining_time = row['elapsed_time'] % time_window
            if remaining_time > 0:
                # Store the remaining part for the next iteration
                buffer_row = row.copy()
                buffer_row['elapsed_time'] = remaining_time
                buffer_row['issue_tick'] = row['issue_tick'] + (full_windows * time_window)
                buffer_row['callback_tick'] = buffer_row['issue_tick'] + remaining_time
            else:
                buffer_row = None
    
    # Don't forget to add the last buffer row if it exists
    if buffer_row is not None:
        new_rows.append(buffer_row)
    
    # Create the new dataframe
    if new_rows:
        df_new = pd.DataFrame(new_rows)
        # Sort the final dataframe by issue_tick
        df_new = df_new.sort_values('issue_tick').reset_index(drop=True)
        return df_new
    else:
        # Return an empty dataframe with the same columns as the input
        return pd.DataFrame(columns=df.columns)


In [ ]:
df = pd.read_csv("output/GPT_3_1300M/2D_Torus/4_8_2_1_0_roofline.csv")
df_single_npu = add_comm_points(df, npu = 0)
df_single_npu_2 = expand_df_and_average(df_single_npu, time_window= 30000)
plot_3d_roofline(df_single_npu.loc[:, ["perf", "operational_intensity", "issue_tick", "node_id"]].drop_duplicates(), 2000, 300)

In [ ]:
import pandas as pd
import numpy as np
import altair as alt

def plot_roofline_avg(df, beta=2000, pi=300, window_size=100000):
    beta = beta * (10 ** 9)
    pi = pi * (10 ** 12)
    I_c = pi / beta
    P_c = pi

    log_Ic = np.log10(I_c)
    log_xmin = log_Ic - (log_Ic - 0) * 1.5
    log_xmax = log_Ic + (log_Ic - log_xmin) / 2
    x_vals = np.logspace(log_xmin, log_xmax, 100)

    roofline_data = pd.DataFrame({
        'operational_intensity': x_vals,
        'beta_line': beta * x_vals,
        'pi_line': [pi] * len(x_vals)
    })

    # Base scatter plot for original points
    chart = alt.Chart(df).mark_circle(color='blue').encode(
        x=alt.X('operational_intensity:Q', scale=alt.Scale(type='log'), title='Operational Intensity (FLOPs/byte)'),
        y=alt.Y('perf:Q', scale=alt.Scale(type='log'), title='Performance (FLOPs/sec)'),
        tooltip=list(df.columns)
    )

    # Moving average over time/cycles
    df_sorted = df.sort_values(by='issue_tick')  # replace with 'time' if that's your column
    bins = (df_sorted['issue_tick'] // window_size) * window_size

    df_avg = df_sorted.groupby(bins).agg({
        'operational_intensity': 'mean',
        'perf': 'mean'
    }).reset_index()

    df_avg.rename(columns={'issue_tick': 'issue_tick'}, inplace=True)

    avg_points = alt.Chart(df_avg).mark_point(
        color='orange',
        size=150,
        shape='diamond'
    ).encode(
        x='operational_intensity:Q',
        y='perf:Q',
        tooltip=list(df_avg.columns)
    )

    # Roofline lines
    beta_line = alt.Chart(roofline_data).mark_line(color='red').encode(
        x='operational_intensity:Q',
        y='beta_line:Q'
    )

    pi_line = alt.Chart(roofline_data).mark_line(color='green').encode(
        x='operational_intensity:Q',
        y='pi_line:Q'
    )

    # Intersection point
    intersect_point = pd.DataFrame({
        'operational_intensity': [I_c],
        'perf': [P_c]
    })

    intersection = alt.Chart(intersect_point).mark_point(color='black', shape='cross', size=200).encode(
        x='operational_intensity:Q',
        y='perf:Q'
    )

    # Combine charts
    final_chart = (chart + avg_points + beta_line + pi_line + intersection).interactive()

    display(final_chart)


In [ ]:
import altair as alt
import numpy as np
import pandas as pd


def plot_roofline_time(df, beta=2000, pi=300):
    # Compute intersection point
    beta = beta * (10**9)
    pi = pi  # TFLOPs/sec
    I_c = (pi * (10**12)) / beta
    P_c = pi

    # Create a new column labeling points based on operational intensity threshold
    df["oi_category"] = np.select(
        [df["operational_intensity"] == 0, df["operational_intensity"] < I_c],
        ["Zero OI", "Memory bound"],
        default="Compute bound",
    )

    y_min = 0
    y_max = df["perf"].max() * 1.1

    # Base scatter plot: x = issue_tick, y = perf, color = operational_intensity

    chart = (
        alt.Chart(df)
        .mark_circle(size=100)
        .encode(
            x=alt.X("issue_tick:Q", title="Issue Time (cycles)"),
            y=alt.Y("perf:Q", title="Performance (TFLOPs/sec)"),
            color=alt.Color(
                "oi_category:N",
                legend=None,
                scale=alt.Scale(
                    domain=["Zero OI", "Memory bound", "Compute bound"],
                    range=["rgba(0,0,0,0.3)", "red", "steelblue"],
                ),
            ),
            tooltip=[
                alt.Tooltip("node_id:N", title="Node ID"),
                alt.Tooltip("node_name:N", title="Node Name"),
                alt.Tooltip("perf:Q", title="Performance (TFLOPs/sec)", format=".2f"),
                alt.Tooltip("issue_tick:Q", title="Issue Time (cycles)"),
                alt.Tooltip(
                    "operational_intensity:Q",
                    title="Operational Intensity",
                    format=".2f",
                ),
            ],
        )
        .properties(
            title="Performance vs Issue Time (colored by Operational Intensity threshold)",
            width=800,
            height=400,
        )
    )

    line = (
        alt.Chart(df)
        .mark_line()
        .encode(x=alt.X("issue_tick:Q"), y=alt.Y("perf:Q"), tooltip=list(df.columns))
    )

    pi_line = (
        alt.Chart(
            pd.DataFrame(
                {
                    "perf": [P_c, P_c],
                    "issue_tick": [df["issue_tick"].min(), df["issue_tick"].max()],
                }
            )
        )
        .mark_line(color="green", strokeDash=[5, 5])
        .encode(x="issue_tick:Q", y="perf:Q")
    )

    final_chart = (chart + pi_line + line).interactive()
    display(final_chart)


In [ ]:
df = pd.read_csv("output/GPT_3_1300M/2D_Torus/1_8_2_4_0_roofline.csv")
df_single_npu = add_comm_points(df, npu = 0)
df_single_npu_2 = expand_df_and_average(df_single_npu, time_window= 100000)
plot_roofline_time(df_single_npu_2.loc[:, ["perf", "operational_intensity", "issue_tick", "node_id", "node_name"]].drop_duplicates(), 2000, 300)

In [ ]:
import altair as alt
import numpy as np
import pandas as pd


def plot_roofline_time_series(df, beta=2000, pi=300):
    beta = beta * (10 ** 9)
    pi = pi
    I_c = pi / beta
    P_c = pi

    x_min = max(0, df['operational_intensity'].min())
    x_max = df['operational_intensity'].max() * 1.1
    x_vals = np.linspace(x_min, x_max, 200)

    roofline_data = pd.DataFrame({
        'operational_intensity': x_vals,
        'beta_line': (beta * x_vals) / 1e12,
        'pi_line': [pi] * len(x_vals)
    })

    # Unique issue_tick values
    unique_times = sorted(df['issue_tick'].unique())
    num_times = len(unique_times)

    # Param for current index
    time_index = alt.param(
        name='time_index',
        bind=alt.binding_range(min=0, max=num_times - 1, step=1, name='Time Index:'),
        value=0
    )

    # Map index to actual time via calculate
    df['time_index'] = df['issue_tick'].apply(lambda t: unique_times.index(t))

    points = alt.Chart(df).mark_circle(size=100).encode(
        x=alt.X('operational_intensity:Q', title='Operational Intensity (FLOPs/byte)'),
        y=alt.Y('perf:Q', title='Performance (TFLOPs/sec)'),
        color=alt.Color('node_name:N', legend=alt.Legend(orient='top-right')),
        tooltip=['node_id', 'node_name', 'perf', 'operational_intensity', 'issue_tick']
    ).transform_filter(
        "datum.time_index == time_index"
    )

    beta_line = alt.Chart(roofline_data).mark_line(color='red').encode(
        x='operational_intensity:Q',
        y='beta_line:Q'
    )

    pi_line = alt.Chart(roofline_data).mark_line(color='green').encode(
        x='operational_intensity:Q',
        y='pi_line:Q'
    )

    intersect_point = pd.DataFrame({
        'operational_intensity': [I_c],
        'perf': [P_c]
    })
    intersection = alt.Chart(intersect_point).mark_point(color='black', shape='cross', size=200).encode(
        x='operational_intensity:Q',
        y='perf:Q'
    )

    # Buttons for navigation
    next_button = alt.binding_radio(options=[1], name='Next ▶️ ')
    prev_button = alt.binding_radio(options=[-1], name='◀️ Prev ')

    nav_param = alt.param(value=0, name='nav_step')

    chart_with_nav = (points + beta_line + pi_line + intersection).add_params(
        time_index, nav_param
    ).transform_calculate(
        time_label=f"'Time: ' + {unique_times}[datum.time_index]"
    ).properties(
        title='Roofline Model (Navigate with Next / Prev)',
        width=650,
        height=450
    ).interactive()

    display(chart_with_nav)

In [ ]:
df = pd.read_csv("output/GPT_3_1300M/2D_Torus/4_8_2_1_0_roofline.csv")

df_single_npu = add_comm_points(df, npu = 0)
df_single_npu_2 = expand_df_and_average(df_single_npu, time_window= 100000)
plot_roofline_time_series(df_single_npu_2.loc[:, ["perf", "operational_intensity", "issue_tick", "node_id", "node_name"]].drop_duplicates(), 2000, 300)

In [ ]:
df = pd.read_csv("output/GPT_3_1300M/2D_Torus/4_1_16_1_0_roofline.csv")
df_single_npu = add_comm_points(df, npu = 0)
plot_roofline(df_single_npu.loc[:, ["perf", "operational_intensity", "issue_tick", "node_id", "node_name"]].drop_duplicates(), 2000, 300)

In [ ]:
def plot_roofline_timestep(df, timestep, beta=2000, pi=300):

    df = df[df['issue_tick'] == timestep]
    
    # Compute intersection point
    I_c = (pi * 1e12) / (beta * 1e9)
    P_c = pi

    x_min = min(0, df['operational_intensity'].min())
    x_max = max(df['operational_intensity'].max(), 100) * 1.1 # Add 10% headroom
    x_vals = np.linspace(x_min, x_max, 200)


    # Create dataframes for the roofline model lines
    roofline_data = pd.DataFrame({
        'operational_intensity': x_vals,
        'beta_line': (beta * x_vals) * 1e-3,
        'pi_line': [pi] * len(x_vals)
    })

    # Base scatter plot
    chart = alt.Chart(df).mark_circle().encode(
        x=alt.X('operational_intensity:Q', title='Operational Intensity (FLOPs/byte)'),
        y=alt.Y('perf:Q', title='Performance (TFLOPs/sec)'),
        size=alt.value(100),
        tooltip=list(df.columns)
    ).properties(
        title='Roofline Model: Performance vs Operational Intensity',
        width=600,
        height=400
    )

    # Bandwidth line (sloped)
    beta_line = alt.Chart(roofline_data).mark_line(color='red').encode(
        x='operational_intensity:Q',
        y='beta_line:Q'
    )

    # Peak performance line (horizontal)
    pi_line = alt.Chart(roofline_data).mark_line(color='green').encode(
        x='operational_intensity:Q',
        y='pi_line:Q'
    )

    # Intersection point marker
    intersect_point = pd.DataFrame({
        'operational_intensity': [I_c],
        'perf': [P_c]
    })
    intersection = alt.Chart(intersect_point).mark_point(color='black', shape='cross', size=200).encode(
        x='operational_intensity:Q',
        y='perf:Q'
    )

    final_chart = (chart + beta_line + pi_line).interactive()
    display(final_chart)

In [ ]:
df = pd.read_csv("output/GPT_3_1300M/2D_Torus/1_8_2_4_0_roofline.csv")
df_single_npu = add_comm_points(df, npu = 0)
#df_single_npu = expand_df_and_average(df_single_npu, time_window= 10000)
timesteps = df_single_npu["issue_tick"].unique()
plot_roofline_timestep(df_single_npu.loc[:, ["perf", "operational_intensity", "issue_tick", "node_id", "node_name"]].drop_duplicates(), timesteps[2], 2000, 300)

In [ ]:
df = pd.read_csv("output/GPT_3_1300M/2D_Torus/4_2_2_4_0_roofline.csv")
df_single_npu = add_comm_points(df, npu = 0)
plot_roofline(df_single_npu.loc[:, ["perf", "operational_intensity", "issue_tick", "node_id", "node_name"]].drop_duplicates(), 2000, 300)

In [ ]:
df = pd.read_csv("output/GPT_3_1300M/2D_Torus/4_8_2_1_0_roofline.csv")
df_single_npu = df.query(f"sys_id == {0}")
plot_roofline(df_single_npu.loc[:, ["perf", "operational_intensity"]].drop_duplicates())

In [ ]:
df = pd.read_csv("output/GPT_3_1300M/2D_Torus/16_1_1_4_0_roofline.csv")
df_single_npu = df.query(f"sys_id == {0}")
plot_roofline(df_single_npu.loc[:, ["perf", "operational_intensity"]].drop_duplicates())

In [ ]:
for arch in ["2D_Torus", "3D_Torus", "DGX1", "DGX_H100", "Dragonfly", "FullyConnected", "Ring", "Switch"]:
    df = pd.read_csv("output/GPT_3_1300M/3D_Torus/1_8_2_4_0_roofline.csv")
    print(f"{arch} 1_8_2_4_0")
    df_single_npu = df.query(f"sys_id == {0}")
    plot_roofline(df_single_npu.loc[:, ["perf", "operational_intensity"]].drop_duplicates())


# Questions / next steps
* Do the computation and communication cycles match the ones obtained at the end of the output?
* Can we get the sizes of the **computation nodes** in flops or something? Check the [chatGPT response](https://chatgpt.com/c/67d88c68-3510-8007-bf07-d8712511c914)
* Can we get the sizes of the **communication nodes** in GB or something? Check the [chatGPT response](https://chatgpt.com/c/67d88c68-3510-8007-bf07-d8712511c914)
* Can we get some kind of direction on what is the bottleneck?
    * maybe a classification of the traces in something like:
        * memory constrained
        * compute constrained
        * comm constrained
 
    * separate in colors:
        * comp
        * comm
        * idle
        * then have an indicator across time, indicating if we have or not comp and comm. And see idle

* does astra sim have "resolution" on the memory accessess of data (L1, L2 cache etc)?
* there is a way to simulate an HBM with memory and latency. Does not include the size of the memory
* Since we are using STG, we get 2 types of nodes. There is a 3rd catgory of nodes: memory load / store. It would use this HBM model.

Jordi Ros
* can we modify incrementally with a delta the compute resources, memory etc, and get a sense of the "derivative" at each timestep.
Corti
* There is a technique called "design of experiments" in statistics that does this

# Next steps (from meeting)
* how complex would be to add memory to astrasim so it can be aware of OOM issues.
* The roofline is a bandwidth model. We can play incrementally with bandwidth and then see how the system would react to that, and see the gradient of modifying bandwidth.
* visualize for each gpu, plot a node as a point in the roofline model. Then we would have N points here and see if we are bottlenecked by comp, bandwidth or what.


* locate comm in the trace 

# Next steps 2025-03-27

2 directions
1. keep modelling memory
2. add NS3 or G2 for network
3. extend the STG to ZeRO family and ZeRO++
    * we will need to add memory nodes that are not present in STG
    * see how many more communication nodes shall we add to STG traces in order to 
4. increase speed of trace generation
5. the overall solver
6. metrics to quantify bottlenecks:
    * how idle is the computation, see proportions etc, shall we optimize the max, the average, etc